# pytest

실제 결함이 있을 때 실패하는 테스트로 확인하고, 수정 뒤 통과하는 테스트로 어떻게 남길까요?

## 실행 환경 준비

Python 3.12와 pytest 9.1.1을 사용합니다. 테스트 파일은 임시 폴더에 만들며, pytest는 현재 커널의 Python으로 실행합니다.

In [1]:
# 교재와 같은 pytest 버전을 현재 커널에 설치합니다.
%pip install -q pytest==9.1.1

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 테스트 파일을 임시 폴더에 두고 외부 pytest 플러그인의 영향을 제외합니다.
import os
from pathlib import Path
from subprocess import run
from sys import executable
from tempfile import TemporaryDirectory

workspace = TemporaryDirectory(prefix="ch02_08_")
test_dir = Path(workspace.name)
pytest_env = os.environ.copy()
pytest_env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
print("pytest 실습 폴더 준비 완료")

pytest 실습 폴더 준비 완료


## 기본 실행

가장 작은 테스트 함수는 이름을 `test_`로 시작하고 매개변수 없이 정의합니다. pytest가 이 함수를 호출하면 `assert total(1200, 800) == 2000`이 왼쪽 함수의 실제 반환값과 오른쪽 기대값을 비교합니다. 참이면 테스트가 통과하며, pytest 프로세스는 종료 코드 0과 `1 passed` 요약을 반환합니다.

In [3]:
# 지정한 파일만 새 프로세스에서 테스트하고 종료 코드를 확인합니다.
def run_pytest(folder: Path, target: str):
    return run(
        [executable, "-m", "pytest", "-q", "--tb=short", target],
        cwd=folder, env=pytest_env, capture_output=True, text=True
    )

basic_test = '''def test_total() -> None:
    assert 1200 + 800 == 2000
'''
(test_dir / "test_basic.py").write_text(basic_test, encoding="utf-8")
basic_result = run_pytest(test_dir, "test_basic.py")
print(basic_result.stdout.strip())
assert basic_result.returncode == 0

.                                                                        [100%]
1 passed in 0.00s


## 중간 결과 관찰

`@pytest.mark.parametrize(("price", "count", "expected"), [...])`는 바로 아래 테스트 함수에 적용하는 pytest 전용 표기입니다. 일반적인 데코레이터 문법은 09절에서 배웁니다. 여기서는 목록의 각 튜플을 하나의 입력 묶음으로 읽습니다. 첫 튜플 `(1000, 1, 1000)`은 `price`, `count`, `expected`에 차례로 전달되고 pytest가 `test_line_total()`을 세 번 호출합니다.

In [4]:
# 입력과 기대값 세 묶음으로 같은 곱셈 검사를 반복합니다.
param_test = '''import pytest

@pytest.mark.parametrize(("price", "count", "expected"), [
    (1000, 1, 1000), (1200, 2, 2400), (500, 0, 0),
])
def test_line_total(price: int, count: int, expected: int) -> None:
    assert price * count == expected
'''
(test_dir / "test_param.py").write_text(param_test, encoding="utf-8")
param_result = run_pytest(test_dir, "test_param.py")
print(param_result.stdout.strip())
assert param_result.returncode == 0

...                                                                      [100%]
3 passed in 0.00s


## 값 하나를 바꿔 비교

`param_test.replace(기존 문자열, 새 문자열)`은 테스트 코드 문자열에서 기존 입력 목록의 일부를 새 입력이 추가된 문자열로 바꾸어 새 문자열을 반환합니다. 원래 `param_test`는 유지되고 반환값이 `expanded`에 저장됩니다. `(900, 3, 2700)` 한 묶음만 추가되므로 함수 구현과 비교식은 유지되고 pytest의 호출 횟수가 세 번에서 네 번으로 늘어납니다.

In [5]:
# 900원 상품 세 개의 사례만 추가하여 총 네 사례를 검사합니다.
expanded = param_test.replace("(500, 0, 0),", "(500, 0, 0), (900, 3, 2700),")
(test_dir / "test_param.py").write_text(expanded, encoding="utf-8")
expanded_result = run_pytest(test_dir, "test_param.py")
print(expanded_result.stdout.strip())
assert expanded_result.returncode == 0

....                                                                     [100%]
4 passed in 0.00s


## 실패 재현

pytest가 테스트 파일을 읽고 `test_discounted()`를 호출합니다. 테스트는 입력 10,000원을 `discounted()`에 전달해 실제값 9,100원을 받고, 기대값 9,000원과 비교합니다. 비교가 거짓이므로 `1 failed`와 종료 코드 1이 반환됩니다.

In [6]:
# 같은 10,000원 입력에서 잘못된 할인 계산을 테스트로 발견합니다.
buggy_test = '''def discounted(price: int) -> int:
    return int(price * 0.9) + 100

def test_discounted() -> None:
    assert discounted(10000) == 9000
'''
(test_dir / "test_discount.py").write_text(buggy_test, encoding="utf-8")
failed = run_pytest(test_dir, "test_discount.py")
print(failed.stdout.strip())
print("종료 코드:", failed.returncode)
assert failed.returncode == 1

F                                                                        [100%]
=================================== FAILURES ===================================
_______________________________ test_discounted ________________________________
test_discount.py:5: in test_discounted
    assert discounted(10000) == 9000
E   assert 9100 == 9000
E    +  where 9100 = discounted(10000)
=========================== short test summary info ============================
FAILED test_discount.py::test_discounted - assert 9100 == 9000
1 failed in 0.01s
종료 코드: 1


## 수정

`buggy_test.replace("int(price * 0.9) + 100", "int(price * 0.9)")`은 테스트 코드 문자열에서 잘못된 계산식만 바꾼 새 문자열을 반환합니다. 입력 10,000원, 기대값 9,000원, 테스트 함수는 유지됩니다. 수정된 함수가 9,000원을 반환하면 같은 단언이 참이 되어 `1 passed`와 종료 코드 0이 반환됩니다.

In [7]:
# 기대값은 유지하고 계산에 더하던 100원만 제거해 다시 검사합니다.
fixed_test = buggy_test.replace("int(price * 0.9) + 100", "int(price * 0.9)")
(test_dir / "test_discount.py").write_text(fixed_test, encoding="utf-8")
passed = run_pytest(test_dir, "test_discount.py")
print(passed.stdout.strip())
print("종료 코드:", passed.returncode)
assert passed.returncode == 0

.                                                                        [100%]
1 passed in 0.00s
종료 코드: 0
